# ConInstruct sample-local constraint trajectory pilot

This notebook runs Route B: every example gets its own original/new constraint basis. It never averages hidden-state directions across examples.

For the same generated conflict response prefix, we teacher-force four prompts: seed, original, new, and conflict. At each layer and prefix:

- `dynamic_local`: rebuild the two-dimensional basis at that prefix.
- `fixed_prompt`: keep the prompt-end basis fixed and test whether it remains readable.
- `selected_margin > 0`: the partial loading favors the constraint that the behavioral judge says the response followed.
- `subspace_fraction`: fraction of the conflict effect explained by the local original/new span.
- `residual_ratio`: off-subspace interaction; large values are evidence against simple composition.

The default response cap is 4096 tokens. Set it to 8192 for very long CoT if GPU memory and the model context window permit.

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Run once per fresh Colab runtime. This experiment uses Transformers, not vLLM.
%pip install -q -U transformers accelerate pandas pyarrow seaborn scipy tqdm


In [8]:
import os
import platform
import torch
import transformers

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 5.12.1
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU memory GiB: 39.5


## Project path

With the VS Code Colab extension, point `PROJECT_ROOT` at the uploaded/mounted repository. The fallback below uses the Google Drive location used by the earlier hidden-state runs.

In [9]:
from pathlib import Path

candidate_roots = [
    Path('/content/drive/MyDrive/ConInstruct-master'),
    Path('/content/ConInstruct-master'),
    Path.cwd(),
]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / 'datasets/constraint_triplets.jsonl').exists()),
    candidate_roots[0],
)

MODEL = 'Qwen/Qwen3.5-4B'
THINKING_MODE = 'non_thinking'
ENABLE_THINKING = False
LIMIT = 100
LAYERS = '8,16,24,31'
PREFIXES = '0,1,4,16,32,64,128,256,512,1024,2048,4096'
MAX_RESPONSE_TOKENS = 4096  # raise to 8192 for long CoT if the runtime can hold it
FORWARD_BATCH_SIZE = 4       # four counterfactual prompts = one sample per forward pass

MODEL_SAFE = MODEL.replace('/', '__')
RESPONSE_ROOT = PROJECT_ROOT / 'outputs/constraint_triplets' / MODEL / THINKING_MODE / 'conflict'
evaluation_root_candidates = [
    PROJECT_ROOT / 'evaluation_outputs/constraint_triplets' / 'judge_gpt-4o-2024-11-20' / MODEL / THINKING_MODE / 'conflict',
    PROJECT_ROOT / 'evaluation_outputs/constraint_triplets' / MODEL / THINKING_MODE / 'conflict',
]
EVALUATION_ROOT = next(
    (path for path in evaluation_root_candidates if len(list(path.rglob('*.json'))) > 0),
    evaluation_root_candidates[0],
)
OUTPUT_DIR = PROJECT_ROOT / 'analysis/constraint_trajectory' / MODEL_SAFE / THINKING_MODE / f'pilot_{LIMIT}'
SCRIPT = PROJECT_ROOT / 'LLMs/extract_constraint_trajectory_geometry.py'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Script exists:', SCRIPT.exists())
print('Responses:', len(list(RESPONSE_ROOT.rglob('*.json'))), RESPONSE_ROOT)
print('Evaluations:', len(list(EVALUATION_ROOT.rglob('*.json'))), EVALUATION_ROOT)
print('Output:', OUTPUT_DIR)


PROJECT_ROOT: /content/drive/MyDrive/ConInstruct-master
Script exists: True
Responses: 864 /content/drive/MyDrive/ConInstruct-master/outputs/constraint_triplets/Qwen/Qwen3.5-4B/non_thinking/conflict
Evaluations: 0 /content/drive/MyDrive/ConInstruct-master/evaluation_outputs/constraint_triplets/judge_gpt-4o-2024-11-20/Qwen/Qwen3.5-4B/non_thinking/conflict
Output: /content/drive/MyDrive/ConInstruct-master/analysis/constraint_trajectory/Qwen__Qwen3.5-4B/non_thinking/pilot_100


In [10]:
import subprocess
import sys

command = [
    sys.executable,
    str(SCRIPT),
    '--model', MODEL,
    '--triplets', str(PROJECT_ROOT / 'datasets/constraint_triplets.jsonl'),
    '--responses-root', str(RESPONSE_ROOT),
    '--evaluations-root', str(EVALUATION_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--layers', LAYERS,
    '--prefixes', PREFIXES,
    '--max-response-tokens', str(MAX_RESPONSE_TOKENS),
    '--forward-batch-size', str(FORWARD_BATCH_SIZE),
    '--enable-thinking', str(ENABLE_THINKING).lower(),
    '--selection', 'stratified',
    '--labels', 'org,new',
    '--limit', str(LIMIT),
    '--resume',
]
print(' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)


/usr/bin/python3 /content/drive/MyDrive/ConInstruct-master/LLMs/extract_constraint_trajectory_geometry.py --model Qwen/Qwen3.5-4B --triplets /content/drive/MyDrive/ConInstruct-master/datasets/constraint_triplets.jsonl --responses-root /content/drive/MyDrive/ConInstruct-master/outputs/constraint_triplets/Qwen/Qwen3.5-4B/non_thinking/conflict --evaluations-root /content/drive/MyDrive/ConInstruct-master/evaluation_outputs/constraint_triplets/judge_gpt-4o-2024-11-20/Qwen/Qwen3.5-4B/non_thinking/conflict --output-dir /content/drive/MyDrive/ConInstruct-master/analysis/constraint_trajectory/Qwen__Qwen3.5-4B/non_thinking/pilot_100 --layers 8,16,24,31 --prefixes 0,1,4,16,32,64,128,256,512,1024,2048,4096 --max-response-tokens 4096 --forward-batch-size 4 --enable-thinking false --selection stratified --labels org,new --limit 100 --resume


CalledProcessError: Command '['/usr/bin/python3', '/content/drive/MyDrive/ConInstruct-master/LLMs/extract_constraint_trajectory_geometry.py', '--model', 'Qwen/Qwen3.5-4B', '--triplets', '/content/drive/MyDrive/ConInstruct-master/datasets/constraint_triplets.jsonl', '--responses-root', '/content/drive/MyDrive/ConInstruct-master/outputs/constraint_triplets/Qwen/Qwen3.5-4B/non_thinking/conflict', '--evaluations-root', '/content/drive/MyDrive/ConInstruct-master/evaluation_outputs/constraint_triplets/judge_gpt-4o-2024-11-20/Qwen/Qwen3.5-4B/non_thinking/conflict', '--output-dir', '/content/drive/MyDrive/ConInstruct-master/analysis/constraint_trajectory/Qwen__Qwen3.5-4B/non_thinking/pilot_100', '--layers', '8,16,24,31', '--prefixes', '0,1,4,16,32,64,128,256,512,1024,2048,4096', '--max-response-tokens', '4096', '--forward-batch-size', '4', '--enable-thinking', 'false', '--selection', 'stratified', '--labels', 'org,new', '--limit', '100', '--resume']' returned non-zero exit status 1.

## Geometry quality checks

Do not interpret partial coefficients when the two local basis vectors are nearly collinear. The main plots below use `basis_condition <= 20`.

In [ ]:
import json
import numpy as np
import pandas as pd

results = pd.read_csv(OUTPUT_DIR / 'trajectory_geometry.csv')
metadata = json.loads((OUTPUT_DIR / 'metadata.json').read_text())

print('Rows:', len(results))
print('Unique samples:', results[['conflict_type_idx', 'sample_id']].drop_duplicates().shape[0])
print('Label counts:')
display(results[['conflict_type_idx', 'sample_id', 'behavior_label_name']].drop_duplicates()['behavior_label_name'].value_counts())
print('Basis conditioning by mode:')
display(results.groupby('basis_mode')['basis_condition'].describe(percentiles=[.5, .75, .9, .95]))

quality = results[
    results['basis_condition'].notna()
    & (results['basis_condition'] <= 20)
    & results['selected_margin'].notna()
].copy()
print('Rows retained after condition-number filter:', len(quality), '/', len(results))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
checkpoints = quality[~quality['is_final'] | quality['prefix_tokens'].isin([0,1,4,16,32,64,128,256,512,1024,2048,4096])].copy()
checkpoints['log2_prefix_plus_1'] = np.log2(checkpoints['prefix_tokens'] + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
specs = [
    ('selected_margin', 'Behavior-selected partial-loading margin'),
    ('subspace_fraction', 'Conflict effect explained by org/new span'),
    ('residual_ratio', 'Off-subspace interaction ratio'),
    ('selected_alignment_margin', 'Behavior-selected cosine margin'),
]
for ax, (metric, title) in zip(axes.ravel(), specs):
    sns.lineplot(
        data=checkpoints,
        x='log2_prefix_plus_1',
        y=metric,
        hue='basis_mode',
        style='behavior_label_name',
        estimator='mean',
        errorbar=('ci', 95),
        ax=ax,
    )
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('Response prefix tokens (log2(tokens + 1))')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'pilot_trajectory_overview.png', dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
from scipy.stats import wilcoxon

id_columns = ['conflict_type_idx', 'sample_id', 'layer', 'basis_mode']
prompt = quality[quality['is_prompt']].drop_duplicates(id_columns).set_index(id_columns)
final = quality[quality['is_final']].drop_duplicates(id_columns).set_index(id_columns)
paired = prompt[['selected_margin', 'subspace_fraction', 'residual_ratio']].join(
    final[['selected_margin', 'subspace_fraction', 'residual_ratio']],
    lsuffix='_prompt',
    rsuffix='_final',
    how='inner',
).reset_index()

paired['selected_margin_delta'] = paired['selected_margin_final'] - paired['selected_margin_prompt']
paired['subspace_fraction_delta'] = paired['subspace_fraction_final'] - paired['subspace_fraction_prompt']
paired['residual_ratio_delta'] = paired['residual_ratio_final'] - paired['residual_ratio_prompt']
display(paired.groupby(['basis_mode', 'layer'])[[
    'selected_margin_delta', 'subspace_fraction_delta', 'residual_ratio_delta'
]].agg(['count', 'mean', 'median']))

test_rows = []
for (basis_mode, layer), group in paired.groupby(['basis_mode', 'layer']):
    values = group['selected_margin_delta'].dropna().to_numpy()
    statistic, pvalue = wilcoxon(values) if len(values) and np.any(values != 0) else (np.nan, np.nan)
    test_rows.append({
        'basis_mode': basis_mode,
        'layer': layer,
        'n': len(values),
        'mean_delta': values.mean() if len(values) else np.nan,
        'median_delta': np.median(values) if len(values) else np.nan,
        'wilcoxon_p': pvalue,
    })
endpoint_tests = pd.DataFrame(test_rows)
endpoint_tests.to_csv(OUTPUT_DIR / 'prompt_to_final_tests.csv', index=False)
display(endpoint_tests)


In [ ]:
# Label-shuffle control: does final geometry align with the actual selected constraint
# more than with labels shuffled within conflict type?
rng = np.random.default_rng(42)
final_dynamic = quality[
    quality['is_final'] & (quality['basis_mode'] == 'dynamic_local')
].copy()

def margin_from_labels(frame, labels):
    return np.where(
        labels == 'new',
        frame['new_partial'].to_numpy() - frame['org_partial'].to_numpy(),
        frame['org_partial'].to_numpy() - frame['new_partial'].to_numpy(),
    )

permutation_rows = []
for layer, layer_frame in final_dynamic.groupby('layer'):
    observed = layer_frame['selected_margin'].mean()
    null = []
    for _ in range(2000):
        shuffled = layer_frame.groupby('conflict_type_idx')['behavior_label_name'].transform(
            lambda values: rng.permutation(values.to_numpy())
        )
        null.append(margin_from_labels(layer_frame, shuffled.to_numpy()).mean())
    null = np.asarray(null)
    pvalue = (1 + np.sum(null >= observed)) / (len(null) + 1)
    permutation_rows.append({
        'layer': layer,
        'n': len(layer_frame),
        'observed_mean_selected_margin': observed,
        'null_mean': null.mean(),
        'null_std': null.std(ddof=1),
        'one_sided_p': pvalue,
    })
permutation_results = pd.DataFrame(permutation_rows)
permutation_results.to_csv(OUTPUT_DIR / 'final_label_permutation.csv', index=False)
display(permutation_results)


## Pilot interpretation

A publishable signal is not merely that the two-dimensional span reconstructs the conflict state. The stronger result is a behavior-linked trajectory:

1. `selected_margin` becomes more positive as generation proceeds and beats the within-conflict-type label permutation.
2. The effect appears across several middle/late layers and conflict types rather than one cherry-picked layer.
3. `dynamic_local` remains informative after filtering ill-conditioned bases.
4. `fixed_prompt` tells us whether the prompt-end geometry is transported. Its failure does not invalidate Route B; it means the local coordinate system rotates during generation.
5. A large `residual_ratio` is itself meaningful: conflict resolution is not a linear sum of the two individually induced constraint effects.